# Missing Values Imputation dan Normalisasi Data pada Dataset Titanic

---

# 1. Missing Values Imputation dengan Metode WKNN

## 1.1 Pengertian Missing Values

Missing values adalah kondisi dimana suatu dataset memiliki nilai atribut yang tidak terisi (kosong). Hal ini dapat terjadi karena berbagai faktor seperti kesalahan pencatatan, data yang tidak tersedia, atau proses pengumpulan data yang tidak lengkap.

Keberadaan missing values dapat menurunkan kualitas data dan memengaruhi hasil analisis. Oleh karena itu, diperlukan teknik untuk menangani masalah tersebut, salah satunya adalah **imputasi data**.

---

## 1.2 Metode Weighted K-Nearest Neighbor (WKNN)

Metode WKNN merupakan pengembangan dari metode KNN (K-Nearest Neighbor). Metode ini digunakan untuk memperkirakan nilai yang hilang berdasarkan kemiripan data dengan data lain yang lengkap.

Perbedaan utama:

* KNN → semua tetangga memiliki bobot yang sama
* WKNN → tetangga yang lebih dekat memiliki bobot lebih besar

Dengan demikian, data yang lebih mirip akan memberikan kontribusi yang lebih besar dalam proses imputasi.

---

## 1.3 Langkah-langkah Metode WKNN

1. Mengidentifikasi data yang memiliki missing value
2. Menentukan fitur yang digunakan sebagai pembanding
3. Melakukan normalisasi data
4. Menghitung jarak antar data
5. Menentukan K tetangga terdekat
6. Menghitung bobot masing-masing tetangga
7. Menghitung nilai imputasi

---

## 1.4 Rumus yang Digunakan

### a. Normalisasi (Min-Max)

$$
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

### b. Bobot (Weight)

$$
w_i = \frac{1}{d_i}
$$

### c. Imputasi

$$
\hat{x} = \frac{\sum_{i=1}^{k} w_i x_i}{\sum_{i=1}^{k} w_i}
$$

Keterangan:

* $d_i$ = jarak data ke-i terhadap data target
* $w_i$ = bobot
* $x_i$ = nilai atribut dari tetangga

---

# 1.5 Dataset yang Digunakan

Dataset yang digunakan adalah **Titanic Dataset** yang berisi data penumpang kapal Titanic.

Dataset ini memiliki sekitar **891 data** dengan beberapa atribut penting, antara lain:

| Kolom  | Deskripsi               |
| ------ | ----------------------- |
| Age    | Umur penumpang          |
| Fare   | Harga tiket             |
| Pclass | Kelas penumpang         |
| SibSp  | Jumlah saudara/pasangan |
| Parch  | Jumlah keluarga         |

Dalam dataset ini:

* Kolom **Age** memiliki banyak missing values
* Kolom **Cabin** memiliki terlalu banyak missing sehingga tidak digunakan

---

# 1.6 Perhitungan WKNN Secara Manual

## 1. Menentukan Missing Value

Misalkan terdapat data berikut:

| Data | Fare | Age |
| ---- | ---- | --- |
| A    | 10   | 20  |
| B    | 20   | 25  |
| C    | 30   | ?   |

Nilai Age pada data C tidak diketahui dan akan dihitung menggunakan WKNN.

---

## 2. Melakukan Normalisasi Data

Normalisasi digunakan agar semua fitur memiliki skala yang sama sehingga tidak ada fitur yang mendominasi.

Rumus:
$$
x' = \frac{x - min}{max - min}
$$

---

## 3. Menghitung Jarak dan Similarity

Jarak:

* C ke A = 20
* C ke B = 10

Similarity:
$$
S_i = \frac{1}{d_i}
$$

* A = 0.05
* B = 0.1

---

## 4. Menghitung Penyebut

$$
0.05 + 0.1 = 0.15
$$

---

## 5. Menghitung Pembilang

$$
(0.05 \cdot 20) + (0.1 \cdot 25) = 3.5
$$

---

## 6. Menghitung Nilai Imputasi

$$
\hat{x} = \frac{3.5}{0.15} = 23.33
$$

Nilai tersebut digunakan untuk menggantikan missing value.

---



# 1.7 Implementasi WKNN Menggunakan Python


In [21]:

## Import Library

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

## Membaca Dataset


In [22]:

df = pd.read_csv("Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



## Menentukan Missing Value


In [23]:


df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


##Menentukan fitur


In [24]:
features = ["Fare", "Pclass", "SibSp", "Parch"]
data = df[["Age"] + features].copy()

print("Fitur yang digunakan untuk menghitung kemiripan:")
print(features)

print("\nData yang akan digunakan:")
print(data.head())

Fitur yang digunakan untuk menghitung kemiripan:
['Fare', 'Pclass', 'SibSp', 'Parch']

Data yang akan digunakan:
    Age     Fare  Pclass  SibSp  Parch
0  22.0   7.2500       3      1      0
1  38.0  71.2833       1      1      0
2  26.0   7.9250       3      0      0
3  35.0  53.1000       1      1      0
4  35.0   8.0500       3      0      0


## Normalisasi Data


In [25]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled = pd.DataFrame(
    scaler.fit_transform(data[features]),
    columns=features,
    index=data.index
)

scaled_data = pd.concat([data[["Age"]], scaled], axis=1)

print("Data setelah standardisasi:")
print(scaled_data.head())

Data setelah standardisasi:
    Age      Fare    Pclass     SibSp     Parch
0  22.0 -0.502445  0.827377  0.432793 -0.473674
1  38.0  0.786845 -1.566107  0.432793 -0.473674
2  26.0 -0.488854  0.827377 -0.474545 -0.473674
3  35.0  0.420730 -1.566107  0.432793 -0.473674
4  35.0 -0.486337  0.827377 -0.474545 -0.473674


## Fungsi WKNN


In [26]:



def wknn(df, target, features, k=3):
    df = df.copy()

    complete = df[df[target].notna()]
    missing = df[df[target].isna()]

    for idx in missing.index:
        row = df.loc[idx, features]

        distances = []
        for i in complete.index:
            dist = np.linalg.norm(row - complete.loc[i, features])
            distances.append((i, dist))

        neighbors = sorted(distances, key=lambda x: x[1])[:k]

        numerator = 0
        denominator = 0

        for i, d in neighbors:
            weight = 1 if d == 0 else 1/d
            numerator += weight * complete.loc[i, target]
            denominator += weight

        df.loc[idx, target] = numerator / denominator

    return df

## Melakukan Imputasi


In [27]:

print("Jumlah missing sebelum imputasi:")
print(df["Age"].isna().sum())

result = wknn(scaled_data, "Age", features, k=3)

print("\nJumlah missing setelah imputasi:")
print(result["Age"].isna().sum())

Jumlah missing sebelum imputasi:
177

Jumlah missing setelah imputasi:
0


---

# 2. Normalisasi Data

## 2.1 Pengertian

Normalisasi adalah proses transformasi data numerik ke dalam skala tertentu agar setiap atribut memiliki kontribusi yang seimbang dalam analisis.

---

## 2.2 Tujuan Normalisasi

* Menghindari dominasi fitur tertentu
* Mempercepat proses pembelajaran model
* Meningkatkan akurasi analisis

---

## 2.3 Jenis-jenis Normalisasi

### 1. Min-Max Normalization

$$
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

Contoh:
Data: 10, 20, 30
Hasil: 0, 0.5, 1

---

### 2. Z-Score Normalization

$$
x' = \frac{x - \mu}{\sigma}
$$

Contoh:

* Mean = 20
* Std = 10

Hasil: -1, 0, 1

---

### 3. Decimal Scaling

$$
x' = \frac{x}{10^j}
$$

---
## 2.4 Implementasi dengan Python

### Min-Max

In [28]:
from sklearn.preprocessing import MinMaxScaler

# Salin data agar aman
df_norm = df.copy()

# Inisialisasi scaler
scaler = MinMaxScaler()

# Normalisasi kolom Fare
df_norm["Fare_minmax"] = scaler.fit_transform(df_norm[["Fare"]])

# Tampilkan hasil
print("Data sebelum dan sesudah normalisasi (Fare):")
print(df_norm[["Fare", "Fare_minmax"]].head(10))

Data sebelum dan sesudah normalisasi (Fare):
      Fare  Fare_minmax
0   7.2500     0.014151
1  71.2833     0.139136
2   7.9250     0.015469
3  53.1000     0.103644
4   8.0500     0.015713
5   8.4583     0.016510
6  51.8625     0.101229
7  21.0750     0.041136
8  11.1333     0.021731
9  30.0708     0.058694


---

### Standardisasi



In [29]:
from sklearn.preprocessing import StandardScaler

# Salin data
df_std = df.copy()

# Inisialisasi scaler
scaler = StandardScaler()

# Standardisasi kolom Fare
df_std["Fare_std"] = scaler.fit_transform(df_std[["Fare"]])

# Tampilkan hasil
print("Perbandingan sebelum dan sesudah standardisasi:")
print(df_std[["Fare", "Fare_std"]].head(10))

Perbandingan sebelum dan sesudah standardisasi:
      Fare  Fare_std
0   7.2500 -0.502445
1  71.2833  0.786845
2   7.9250 -0.488854
3  53.1000  0.420730
4   8.0500 -0.486337
5   8.4583 -0.478116
6  51.8625  0.395814
7  21.0750 -0.224083
8  11.1333 -0.424256
9  30.0708 -0.042956


---

## 2.5 Normalisasi Manual


In [30]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min())

# Salin data
df_manual = df.copy()

# Terapkan normalisasi manual
df_manual["Fare_manual"] = minmax(df_manual["Fare"])

# Tampilkan hasil
print("Perbandingan sebelum dan sesudah normalisasi manual:")
print(df_manual[["Fare", "Fare_manual"]].head(10))

Perbandingan sebelum dan sesudah normalisasi manual:
      Fare  Fare_manual
0   7.2500     0.014151
1  71.2833     0.139136
2   7.9250     0.015469
3  53.1000     0.103644
4   8.0500     0.015713
5   8.4583     0.016510
6  51.8625     0.101229
7  21.0750     0.041136
8  11.1333     0.021731
9  30.0708     0.058694



---

# 3. Kesimpulan

1. Missing values dapat diatasi menggunakan metode WKNN
2. WKNN memberikan bobot lebih besar pada data yang lebih dekat
3. Dataset Titanic cocok digunakan karena memiliki missing value pada Age
4. Normalisasi sangat penting sebelum perhitungan jarak
5. Normalisasi dapat dilakukan menggunakan sklearn maupun manual

---

# 4. Catatan

Pemilihan nilai K dan fitur yang digunakan sangat berpengaruh terhadap hasil imputasi, sehingga perlu disesuaikan dengan kondisi dataset.
